# Loading and Preprocessing

In [14]:
from data_loader import load_data

In [15]:
X_clinical,\
X_clinical_A,\
X_clinical_B,\
X_clinical_Delta,\
targets = load_data()

In [16]:
targets.head()

,surv_bestresponse_car,ae_summ_icans_v2,ae_summ_icans_highestgrade_v2,ae_summ_crs_v2,ae_summ_highestgrade_v2
FTC-UMCG-0001,1.0,1.0,1.0,1.0,1.0
FTC-UMCG-0003,4.0,0.0,0.0,1.0,1.0
FTC-UMCG-0004,4.0,1.0,3.0,1.0,2.0
FTC-UMCG-0005,2.0,1.0,2.0,1.0,1.0
FTC-UMCG-0006,1.0,0.0,0.0,1.0,2.0


In [17]:
datasets = {
    "Clinical": X_clinical,
    "Clinical_A": X_clinical_A,
    "Clinical_B": X_clinical_B,
    "Clinical_Delta": X_clinical_Delta
}

# Feature Selection
Dropping highly correlated clinical and radiomics features using the EDA analysis, as they add little to no new information.

## Highly Correlated Clinical Features  
We're using the EDA because it took the mixed feature types into account when calculating the correlations, and there are only a few features, so we could manually write them. So, it was more straight-forward than redoing it ourselves.

In [18]:
clinical_ftrs_to_drop = ['total_num_priortherapylines_fl',
                'tr_car_bridg_type.factor_None',
                'scr_weight',
                'scr_height',
                'indication_age_60',
                'cli_st_neutrophils',
                'indication_extran_invol',
                'indication_pri_refr',
                'indication_sec_refr'
                ]

In [19]:
for name, df in datasets.items():
    print(f"Before dropping features in {name}: {df.shape}")
    df.drop(columns=clinical_ftrs_to_drop, inplace=True)
    print(f"After dropping features in {name}: {df.shape}")

Before dropping features in Clinical: (85, 35)
After dropping features in Clinical: (85, 26)
Before dropping features in Clinical_A: (85, 134)
After dropping features in Clinical_A: (85, 125)
Before dropping features in Clinical_B: (85, 134)
After dropping features in Clinical_B: (85, 125)
Before dropping features in Clinical_Delta: (85, 134)
After dropping features in Clinical_Delta: (85, 125)


## Highly Correlated Radiomic Features
These are all numerical values, with high dimensionality. The EDA already showed a large number of highly correlated features, so we will write code to automatically keep only one of possibly multiple highly correlated features.

In [20]:
import numpy as np

def keep_uncorrelated_features(df, threshold=0.9):
    # Absolute correlation matrix
    corr = df.corr().abs()
    # Upper triangle (ignore diagonal and lower half), k=1 means we start from the first diagonal above the main diagonal
    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )
    # Find columns with any correlation above threshold
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any() and upper[col].dtype in [np.float64]]
    # Return reduced dataframe
    return df.drop(columns=to_drop)

# Usage
for name, df in datasets.items():
    print(f"Before dropping correlated features in {name}: {df.shape}")

    datasets[name] = keep_uncorrelated_features(df, threshold=0.9)
    print(f"After dropping correlated features in {name}: {datasets[name].shape}")

Before dropping correlated features in Clinical: (85, 26)
After dropping correlated features in Clinical: (85, 26)
Before dropping correlated features in Clinical_A: (85, 125)
After dropping correlated features in Clinical_A: (85, 62)
Before dropping correlated features in Clinical_B: (85, 125)
After dropping correlated features in Clinical_B: (85, 59)
Before dropping correlated features in Clinical_Delta: (85, 125)
After dropping correlated features in Clinical_Delta: (85, 71)


In [21]:
cat_cols = datasets['Clinical'].select_dtypes(exclude='number').columns.tolist()

In [22]:
# Using ColumnTransformer to apply different transformations to numeric and categorical columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Creating a ColumnTransformer to apply StandardScaler to numeric columns and leave categorical columns unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', 'passthrough', cat_cols)],
        remainder= StandardScaler()
    
)

In [23]:
from model_eval import result_viewer

# ICANS 0,1

In [24]:
y = targets['ae_summ_icans_v2']

In [25]:
y.value_counts()

ae_summ_icans_v2
1.0    48
0.0    37
Name: count, dtype: int64

In [26]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6528 ± 0.0519,0.5117 ± 0.1013,0.5686 ± 0.0877,0.4890 ± 0.0273
train_roc_auc,0.7348 ± 0.0107,0.6949 ± 0.0651,0.7180 ± 0.0560,0.7216 ± 0.0363
test_precision,0.6133 ± 0.0499,0.5870 ± 0.1080,0.6151 ± 0.0526,0.6023 ± 0.0730
train_precision,0.6789 ± 0.0434,0.6673 ± 0.0322,0.6806 ± 0.0310,0.6787 ± 0.0509
test_recall,0.8511 ± 0.1119,0.8156 ± 0.1608,0.7867 ± 0.1260,0.8511 ± 0.0923
train_recall,0.7864 ± 0.0201,0.8536 ± 0.0549,0.8173 ± 0.0388,0.8072 ± 0.0560
test_f1,0.7091 ± 0.0575,0.6770 ± 0.1141,0.6799 ± 0.0322,0.7041 ± 0.0767
train_f1,0.7284 ± 0.0332,0.7471 ± 0.0158,0.7421 ± 0.0274,0.7353 ± 0.0353


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4809 ± 0.1215,0.5006 ± 0.0447,0.4851 ± 0.0849,0.5055 ± 0.1453
train_roc_auc,0.7432 ± 0.0459,0.7702 ± 0.0381,0.7438 ± 0.0597,0.7650 ± 0.0213
test_precision,0.5669 ± 0.0787,0.5934 ± 0.0849,0.5692 ± 0.0315,0.5377 ± 0.1529
train_precision,0.7430 ± 0.0790,0.8030 ± 0.1084,0.7347 ± 0.1004,0.7921 ± 0.1045
test_recall,0.7022 ± 0.1952,0.6933 ± 0.2243,0.8778 ± 0.0976,0.6178 ± 0.2717
train_recall,0.8394 ± 0.1510,0.7901 ± 0.1959,0.8684 ± 0.1722,0.7704 ± 0.1671
test_f1,0.6203 ± 0.1257,0.6049 ± 0.1036,0.6871 ± 0.0322,0.5620 ± 0.2086
train_f1,0.7722 ± 0.0379,0.7678 ± 0.0705,0.7727 ± 0.0428,0.7582 ± 0.0353


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5890 ± 0.1326,0.5463 ± 0.0948,0.5450 ± 0.1376,0.5034 ± 0.1101
train_roc_auc,0.8826 ± 0.0248,0.9088 ± 0.0192,0.9209 ± 0.0088,0.9301 ± 0.0143
test_precision,0.6244 ± 0.1183,0.5887 ± 0.0735,0.5785 ± 0.0724,0.5462 ± 0.0615
train_precision,0.7602 ± 0.0324,0.7583 ± 0.0606,0.8147 ± 0.0416,0.7965 ± 0.0350
test_recall,0.6733 ± 0.1537,0.8356 ± 0.1041,0.7911 ± 0.0711,0.7089 ± 0.1156
train_recall,0.8955 ± 0.0579,0.9011 ± 0.0684,0.9111 ± 0.0638,0.9217 ± 0.0469
test_f1,0.6319 ± 0.0829,0.6835 ± 0.0462,0.6671 ± 0.0687,0.6146 ± 0.0794
train_f1,0.8206 ± 0.0259,0.8191 ± 0.0271,0.8575 ± 0.0256,0.8530 ± 0.0194


In [16]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6094 ± 0.1236,0.4929 ± 0.1157,0.5791 ± 0.1703,0.3893 ± 0.0783
train_roc_auc,0.8822 ± 0.0242,0.9899 ± 0.0083,0.9917 ± 0.0058,0.9982 ± 0.0018
test_precision,0.6594 ± 0.1180,0.6353 ± 0.1274,0.6745 ± 0.1797,0.4944 ± 0.0509
train_precision,0.8338 ± 0.0129,0.9640 ± 0.0261,0.9448 ± 0.0284,0.9946 ± 0.0108
test_recall,0.6622 ± 0.1814,0.7067 ± 0.1356,0.6467 ± 0.0980,0.4778 ± 0.1129
train_recall,0.8591 ± 0.0279,0.9688 ± 0.0194,0.9740 ± 0.0232,0.9738 ± 0.0235
test_f1,0.6509 ± 0.1207,0.6594 ± 0.0989,0.6487 ± 0.0978,0.4834 ± 0.0781
train_f1,0.8459 ± 0.0133,0.9664 ± 0.0225,0.9589 ± 0.0206,0.9840 ± 0.0156


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5653 ± 0.1128,0.5385 ± 0.0969,0.4640 ± 0.1036,0.4788 ± 0.0859
train_roc_auc,0.8021 ± 0.0141,0.8078 ± 0.0281,0.8141 ± 0.0167,0.8311 ± 0.0212
test_precision,0.6235 ± 0.0722,0.5744 ± 0.0930,0.5663 ± 0.0219,0.5412 ± 0.1097
train_precision,0.7835 ± 0.0332,0.7826 ± 0.0295,0.8170 ± 0.0770,0.8763 ± 0.0566
test_recall,0.6200 ± 0.1638,0.7489 ± 0.0839,0.7889 ± 0.1241,0.5511 ± 0.2078
train_recall,0.8385 ± 0.0714,0.8588 ± 0.0550,0.8489 ± 0.1421,0.7432 ± 0.1494
test_f1,0.6090 ± 0.0819,0.6487 ± 0.0872,0.6551 ± 0.0539,0.5264 ± 0.1118
train_f1,0.8077 ± 0.0345,0.8180 ± 0.0326,0.8181 ± 0.0431,0.7900 ± 0.0731


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6070 ± 0.1163,0.5756 ± 0.1036,0.6263 ± 0.0859,0.5394 ± 0.0873
train_roc_auc,0.9620 ± 0.0164,0.9719 ± 0.0161,0.9901 ± 0.0079,0.9858 ± 0.0088
test_precision,0.6105 ± 0.0593,0.5763 ± 0.0734,0.5793 ± 0.0148,0.5467 ± 0.0452
train_precision,0.7897 ± 0.0347,0.7731 ± 0.0164,0.8471 ± 0.0339,0.8431 ± 0.0425
test_recall,0.8511 ± 0.0923,0.7978 ± 0.1661,0.8600 ± 0.1744,0.8356 ± 0.1179
train_recall,0.9634 ± 0.0212,0.9895 ± 0.0211,0.9947 ± 0.0105,0.9842 ± 0.0129
test_f1,0.7107 ± 0.0710,0.6510 ± 0.0429,0.6834 ± 0.0550,0.6584 ± 0.0644
train_f1,0.8671 ± 0.0172,0.8677 ± 0.0060,0.9145 ± 0.0161,0.9074 ± 0.0219


## ICANS >=2

In [27]:
y = targets["ae_summ_icans_highestgrade_v2"]

In [28]:
y = y.astype('int8')

In [29]:
y[y<2] = 0
y[y>=2] = 1


In [30]:
y.value_counts()

ae_summ_icans_highestgrade_v2
0    52
1    33
Name: count, dtype: int64

In [31]:
icans_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4968 ± 0.1675,0.4968 ± 0.1986,0.5671 ± 0.1354,0.5897 ± 0.0974
train_roc_auc,0.6853 ± 0.0358,0.6422 ± 0.0577,0.7014 ± 0.0453,0.6574 ± 0.0323
test_precision,0.4833 ± 0.3512,0.4833 ± 0.3432,0.4200 ± 0.3370,0.4833 ± 0.1333
train_precision,0.6040 ± 0.0545,0.7182 ± 0.1118,0.5991 ± 0.0972,0.5470 ± 0.1381
test_recall,0.2429 ± 0.1666,0.1524 ± 0.0911,0.2143 ± 0.1205,0.2143 ± 0.0797
train_recall,0.3177 ± 0.0719,0.2869 ± 0.1774,0.3940 ± 0.1244,0.2587 ± 0.1305
test_f1,0.3104 ± 0.2051,0.2235 ± 0.1322,0.2719 ± 0.1625,0.2908 ± 0.0918
train_f1,0.4106 ± 0.0697,0.3847 ± 0.2009,0.4686 ± 0.1142,0.3427 ± 0.1455


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5267 ± 0.1458,0.5187 ± 0.1260,0.5716 ± 0.0276,0.5601 ± 0.1189
train_roc_auc,0.7192 ± 0.0654,0.7493 ± 0.0587,0.7573 ± 0.0515,0.8088 ± 0.0114
test_precision,0.4500 ± 0.3326,0.3133 ± 0.2035,0.4750 ± 0.3202,0.3911 ± 0.2278
train_precision,0.6726 ± 0.1196,0.7516 ± 0.1314,0.8346 ± 0.1363,0.7393 ± 0.1182
test_recall,0.4619 ± 0.3943,0.3190 ± 0.2398,0.3286 ± 0.2000,0.4048 ± 0.2809
train_recall,0.6487 ± 0.2848,0.5658 ± 0.1803,0.5154 ± 0.1565,0.7117 ± 0.1886
test_f1,0.3705 ± 0.2516,0.3128 ± 0.2178,0.3540 ± 0.1851,0.3779 ± 0.2210
train_f1,0.5969 ± 0.1199,0.6128 ± 0.0963,0.6074 ± 0.0847,0.6938 ± 0.0544


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6454 ± 0.1428,0.4774 ± 0.1757,0.5738 ± 0.1620,0.5302 ± 0.1450
train_roc_auc,0.8903 ± 0.0382,0.9380 ± 0.0180,0.9243 ± 0.0276,0.9401 ± 0.0131
test_precision,0.5000 ± 0.4472,0.4300 ± 0.3995,0.2800 ± 0.2315,0.4000 ± 0.3742
train_precision,0.9139 ± 0.0649,0.9784 ± 0.0265,0.9656 ± 0.0438,0.9414 ± 0.0752
test_recall,0.1524 ± 0.1393,0.1857 ± 0.1895,0.1190 ± 0.1086,0.1857 ± 0.1733
train_recall,0.4322 ± 0.0909,0.5140 ± 0.1457,0.4858 ± 0.1145,0.5840 ± 0.0631
test_f1,0.2227 ± 0.1983,0.2367 ± 0.2252,0.1611 ± 0.1365,0.2423 ± 0.2154
train_f1,0.5793 ± 0.0816,0.6592 ± 0.1253,0.6356 ± 0.0941,0.7149 ± 0.0229


In [21]:
icans_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6151 ± 0.1199,0.5715 ± 0.0777,0.6304 ± 0.0834,0.6393 ± 0.1411
train_roc_auc,0.8985 ± 0.0258,0.9887 ± 0.0085,0.9955 ± 0.0054,0.9931 ± 0.0078
test_precision,0.5362 ± 0.1828,0.4893 ± 0.1704,0.5450 ± 0.3211,0.4450 ± 0.2283
train_precision,0.7789 ± 0.0252,0.9449 ± 0.0401,0.9695 ± 0.0287,0.9763 ± 0.0311
test_recall,0.4714 ± 0.2504,0.4952 ± 0.2230,0.4095 ± 0.2725,0.5619 ± 0.3346
train_recall,0.6672 ± 0.0392,0.9094 ± 0.0510,0.9473 ± 0.0292,0.9328 ± 0.0593
test_f1,0.4815 ± 0.1985,0.4705 ± 0.1580,0.4331 ± 0.2367,0.4899 ± 0.2632
train_f1,0.7183 ± 0.0302,0.9267 ± 0.0447,0.9582 ± 0.0275,0.9535 ± 0.0432


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5135 ± 0.1017,0.5358 ± 0.1184,0.5931 ± 0.1033,0.5502 ± 0.1771
train_roc_auc,0.7474 ± 0.0355,0.8046 ± 0.0441,0.8151 ± 0.0345,0.8095 ± 0.0113
test_precision,0.3394 ± 0.1885,0.5365 ± 0.2433,0.4579 ± 0.1339,0.4605 ± 0.1938
train_precision,0.6400 ± 0.1115,0.7065 ± 0.0653,0.7391 ± 0.0708,0.6875 ± 0.0855
test_recall,0.4571 ± 0.3546,0.4238 ± 0.1048,0.4381 ± 0.2073,0.5143 ± 0.2355
train_recall,0.7969 ± 0.2250,0.6886 ± 0.0760,0.6895 ± 0.0439,0.8322 ± 0.1867
test_f1,0.3632 ± 0.2300,0.4269 ± 0.0243,0.4348 ± 0.1628,0.4756 ± 0.1975
train_f1,0.6742 ± 0.0623,0.6934 ± 0.0490,0.7121 ± 0.0492,0.7277 ± 0.0642


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6324 ± 0.1289,0.5372 ± 0.1384,0.6390 ± 0.0700,0.5908 ± 0.0822
train_roc_auc,0.9629 ± 0.0145,0.9887 ± 0.0049,0.9900 ± 0.0074,0.9957 ± 0.0065
test_precision,0.4000 ± 0.4899,0.0000 ± 0.0000,0.4000 ± 0.4899,0.2667 ± 0.3887
train_precision,0.9846 ± 0.0308,0.9875 ± 0.0250,1.0000 ± 0.0000,1.0000 ± 0.0000
test_recall,0.0905 ± 0.1170,0.0000 ± 0.0000,0.0952 ± 0.1313,0.0952 ± 0.1313
train_recall,0.4091 ± 0.0372,0.5006 ± 0.0951,0.6071 ± 0.0760,0.5772 ± 0.1167
test_f1,0.1460 ± 0.1858,0.0000 ± 0.0000,0.1500 ± 0.2000,0.1400 ± 0.1960
train_f1,0.5764 ± 0.0338,0.6580 ± 0.0872,0.7528 ± 0.0577,0.7246 ± 0.1001


# CRS 0,1

In [32]:
y = targets["ae_summ_crs_v2"]

In [33]:
y.value_counts()

ae_summ_crs_v2
1.0    79
0.0     6
Name: count, dtype: int64

In [34]:
crs_1 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6275 ± 0.2062,0.7496 ± 0.2450,0.6408 ± 0.2956,0.7333 ± 0.1664
train_roc_auc,0.8554 ± 0.0478,0.8698 ± 0.0419,0.8337 ± 0.1065,0.8690 ± 0.0270
test_precision,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9287 ± 0.0232
train_precision,0.9294 ± 0.0059,0.9294 ± 0.0059,0.9294 ± 0.0059,0.9294 ± 0.0059
test_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,0.9875 ± 0.0250
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9568 ± 0.0158
train_f1,0.9634 ± 0.0031,0.9634 ± 0.0031,0.9634 ± 0.0031,0.9634 ± 0.0031


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6121 ± 0.2669,0.6871 ± 0.2745,0.5742 ± 0.2220,0.5054 ± 0.2575
train_roc_auc,0.8940 ± 0.0108,0.8536 ± 0.0444,0.9104 ± 0.0286,0.8946 ± 0.0161
test_precision,0.9269 ± 0.0228,0.9294 ± 0.0235,0.9287 ± 0.0232,0.9262 ± 0.0224
train_precision,0.9378 ± 0.0163,0.9294 ± 0.0059,0.9521 ± 0.0228,0.9433 ± 0.0136
test_recall,0.9625 ± 0.0750,1.0000 ± 0.0000,0.9875 ± 0.0250,0.9500 ± 0.0729
train_recall,0.9937 ± 0.0127,1.0000 ± 0.0000,0.9905 ± 0.0127,0.9937 ± 0.0127
test_f1,0.9427 ± 0.0400,0.9633 ± 0.0129,0.9568 ± 0.0158,0.9362 ± 0.0376
train_f1,0.9647 ± 0.0035,0.9634 ± 0.0031,0.9706 ± 0.0088,0.9676 ± 0.0029


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6933 ± 0.1470,0.7154 ± 0.1850,0.6246 ± 0.2260,0.7008 ± 0.1341
train_roc_auc,0.9086 ± 0.0268,0.9364 ± 0.0651,0.9877 ± 0.0082,0.9471 ± 0.0474
test_precision,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9287 ± 0.0232,0.9294 ± 0.0235
train_precision,0.9294 ± 0.0059,0.9294 ± 0.0059,0.9380 ± 0.0166,0.9294 ± 0.0059
test_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,0.9875 ± 0.0250,1.0000 ± 0.0000
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9568 ± 0.0158,0.9633 ± 0.0129
train_f1,0.9634 ± 0.0031,0.9634 ± 0.0031,0.9679 ± 0.0088,0.9634 ± 0.0031


# CRS >=2

In [35]:
y = targets["ae_summ_highestgrade_v2"]
y = y.astype('int8')
y[y<2] = 0
y[y>=2] = 1

y.value_counts()

ae_summ_highestgrade_v2
0    55
1    30
Name: count, dtype: int64

In [36]:
crs_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6242 ± 0.1412,0.6061 ± 0.1185,0.6485 ± 0.1326,0.5727 ± 0.2027
train_roc_auc,0.6538 ± 0.0398,0.7648 ± 0.0423,0.7277 ± 0.0272,0.7089 ± 0.0552
test_precision,0.2667 ± 0.3887,0.3667 ± 0.2506,0.4733 ± 0.2800,0.4619 ± 0.2337
train_precision,0.7607 ± 0.1646,0.6742 ± 0.0324,0.5617 ± 0.0626,0.5776 ± 0.0866
test_recall,0.0667 ± 0.0816,0.2333 ± 0.1700,0.3333 ± 0.2108,0.2667 ± 0.1333
train_recall,0.2083 ± 0.1179,0.4083 ± 0.1453,0.3167 ± 0.0898,0.2583 ± 0.1130
test_f1,0.1016 ± 0.1260,0.2767 ± 0.1954,0.3729 ± 0.2038,0.3297 ± 0.1675
train_f1,0.3048 ± 0.1481,0.4966 ± 0.1153,0.4000 ± 0.0866,0.3399 ± 0.1212


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5030 ± 0.1081,0.6000 ± 0.0683,0.5636 ± 0.1505,0.5500 ± 0.0904
train_roc_auc,0.7023 ± 0.0319,0.7839 ± 0.0295,0.7912 ± 0.0285,0.8127 ± 0.0180
test_precision,0.1444 ± 0.1975,0.5333 ± 0.3232,0.6333 ± 0.3293,0.3769 ± 0.1937
train_precision,0.7542 ± 0.1359,0.8986 ± 0.0611,0.8613 ± 0.1700,0.7719 ± 0.1507
test_recall,0.1667 ± 0.2108,0.1667 ± 0.1054,0.3000 ± 0.1247,0.4000 ± 0.3091
train_recall,0.2917 ± 0.1514,0.4250 ± 0.1379,0.4667 ± 0.2014,0.5667 ± 0.2085
test_f1,0.1533 ± 0.2018,0.2460 ± 0.1425,0.3723 ± 0.1387,0.3495 ± 0.2074
train_f1,0.3833 ± 0.1522,0.5582 ± 0.1283,0.5518 ± 0.1117,0.6074 ± 0.0845


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5409 ± 0.0795,0.5939 ± 0.0537,0.6273 ± 0.0340,0.6697 ± 0.1976
train_roc_auc,0.7809 ± 0.0428,0.9093 ± 0.0107,0.9127 ± 0.0203,0.9530 ± 0.0070
test_precision,0.2667 ± 0.2759,0.3867 ± 0.3804,0.4333 ± 0.0816,0.4667 ± 0.4522
train_precision,0.8384 ± 0.1335,0.9750 ± 0.0500,0.8858 ± 0.0620,0.9560 ± 0.0577
test_recall,0.1667 ± 0.1826,0.1667 ± 0.1826,0.2333 ± 0.0816,0.1667 ± 0.1491
train_recall,0.2833 ± 0.1568,0.5000 ± 0.0874,0.5167 ± 0.0972,0.4833 ± 0.0425
test_f1,0.2044 ± 0.2193,0.2107 ± 0.2033,0.2989 ± 0.0832,0.2238 ± 0.1961
train_f1,0.3885 ± 0.1620,0.6544 ± 0.0784,0.6478 ± 0.0838,0.6399 ± 0.0371


In [ ]:
icans_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6151 ± 0.1199,0.5715 ± 0.0777,0.6304 ± 0.0834,0.6393 ± 0.1411
train_roc_auc,0.8985 ± 0.0258,0.9887 ± 0.0085,0.9955 ± 0.0054,0.9931 ± 0.0078
test_precision,0.5362 ± 0.1828,0.4893 ± 0.1704,0.5450 ± 0.3211,0.4450 ± 0.2283
train_precision,0.7789 ± 0.0252,0.9449 ± 0.0401,0.9695 ± 0.0287,0.9763 ± 0.0311
test_recall,0.4714 ± 0.2504,0.4952 ± 0.2230,0.4095 ± 0.2725,0.5619 ± 0.3346
train_recall,0.6672 ± 0.0392,0.9094 ± 0.0510,0.9473 ± 0.0292,0.9328 ± 0.0593
test_f1,0.4815 ± 0.1985,0.4705 ± 0.1580,0.4331 ± 0.2367,0.4899 ± 0.2632
train_f1,0.7183 ± 0.0302,0.9267 ± 0.0447,0.9582 ± 0.0275,0.9535 ± 0.0432


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5135 ± 0.1017,0.5358 ± 0.1184,0.5931 ± 0.1033,0.5502 ± 0.1771
train_roc_auc,0.7474 ± 0.0355,0.8046 ± 0.0441,0.8151 ± 0.0345,0.8095 ± 0.0113
test_precision,0.3394 ± 0.1885,0.5365 ± 0.2433,0.4579 ± 0.1339,0.4605 ± 0.1938
train_precision,0.6400 ± 0.1115,0.7065 ± 0.0653,0.7391 ± 0.0708,0.6875 ± 0.0855
test_recall,0.4571 ± 0.3546,0.4238 ± 0.1048,0.4381 ± 0.2073,0.5143 ± 0.2355
train_recall,0.7969 ± 0.2250,0.6886 ± 0.0760,0.6895 ± 0.0439,0.8322 ± 0.1867
test_f1,0.3632 ± 0.2300,0.4269 ± 0.0243,0.4348 ± 0.1628,0.4756 ± 0.1975
train_f1,0.6742 ± 0.0623,0.6934 ± 0.0490,0.7121 ± 0.0492,0.7277 ± 0.0642


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6324 ± 0.1289,0.5372 ± 0.1384,0.6390 ± 0.0700,0.5908 ± 0.0822
train_roc_auc,0.9629 ± 0.0145,0.9887 ± 0.0049,0.9900 ± 0.0074,0.9957 ± 0.0065
test_precision,0.4000 ± 0.4899,0.0000 ± 0.0000,0.4000 ± 0.4899,0.2667 ± 0.3887
train_precision,0.9846 ± 0.0308,0.9875 ± 0.0250,1.0000 ± 0.0000,1.0000 ± 0.0000
test_recall,0.0905 ± 0.1170,0.0000 ± 0.0000,0.0952 ± 0.1313,0.0952 ± 0.1313
train_recall,0.4091 ± 0.0372,0.5006 ± 0.0951,0.6071 ± 0.0760,0.5772 ± 0.1167
test_f1,0.1460 ± 0.1858,0.0000 ± 0.0000,0.1500 ± 0.2000,0.1400 ± 0.1960
train_f1,0.5764 ± 0.0338,0.6580 ± 0.0872,0.7528 ± 0.0577,0.7246 ± 0.1001


# Outcome
CMR vs others

In [37]:
y = targets["surv_bestresponse_car"]

In [38]:
y.value_counts()

surv_bestresponse_car
1.0    63
2.0    12
4.0     8
0.0     2
Name: count, dtype: int64

Early Death: 0, CMR: 1, PMR:2, SD: 3, PD:4

In [39]:
y = y.astype('int8')

In [40]:
y[y != 1] = 0

In [41]:
y.value_counts()

surv_bestresponse_car
1    63
0    22
Name: count, dtype: int64

In [42]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6082 ± 0.1318,0.6608 ± 0.1096,0.5492 ± 0.0856,0.6708 ± 0.0973
train_roc_auc,0.7643 ± 0.0428,0.7340 ± 0.0429,0.7641 ± 0.0331,0.7798 ± 0.0240
test_precision,0.7575 ± 0.0592,0.7622 ± 0.0513,0.7537 ± 0.0319,0.7467 ± 0.0237
train_precision,0.7773 ± 0.0205,0.7920 ± 0.0187,0.7880 ± 0.0280,0.8006 ± 0.0345
test_recall,0.9359 ± 0.0322,0.9526 ± 0.0388,0.9679 ± 0.0393,0.9833 ± 0.0333
train_recall,0.9524 ± 0.0203,0.9801 ± 0.0127,0.9683 ± 0.0156,0.9643 ± 0.0149
test_f1,0.8369 ± 0.0483,0.8451 ± 0.0280,0.8466 ± 0.0214,0.8485 ± 0.0226
train_f1,0.8557 ± 0.0120,0.8759 ± 0.0123,0.8685 ± 0.0164,0.8744 ± 0.0210


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5462 ± 0.0253,0.5140 ± 0.0629,0.4891 ± 0.0917,0.5269 ± 0.0728
train_roc_auc,0.7579 ± 0.0667,0.7991 ± 0.0218,0.7901 ± 0.0330,0.8228 ± 0.0358
test_precision,0.7634 ± 0.0270,0.7600 ± 0.0302,0.7433 ± 0.0082,0.7667 ± 0.0279
train_precision,0.8433 ± 0.0537,0.8605 ± 0.0344,0.8542 ± 0.0343,0.8337 ± 0.0233
test_recall,0.9205 ± 0.0488,0.8551 ± 0.0646,0.9218 ± 0.0487,0.8756 ± 0.1337
train_recall,0.9405 ± 0.0519,0.9365 ± 0.0343,0.9567 ± 0.0502,0.9684 ± 0.0200
test_f1,0.8337 ± 0.0233,0.8038 ± 0.0390,0.8226 ± 0.0231,0.8105 ± 0.0598
train_f1,0.8863 ± 0.0179,0.8957 ± 0.0106,0.9007 ± 0.0110,0.8955 ± 0.0095


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.7127 ± 0.1458,0.6015 ± 0.1544,0.6126 ± 0.1113,0.6454 ± 0.0675
train_roc_auc,0.9072 ± 0.0216,0.9244 ± 0.0169,0.9323 ± 0.0155,0.9410 ± 0.0172
test_precision,0.7400 ± 0.0287,0.7429 ± 0.0440,0.7593 ± 0.0515,0.7279 ± 0.0460
train_precision,0.8067 ± 0.0263,0.8313 ± 0.0259,0.8310 ± 0.0202,0.8257 ± 0.0120
test_recall,0.9513 ± 0.0398,0.9192 ± 0.0528,0.9372 ± 0.0315,0.8590 ± 0.1633
train_recall,0.9882 ± 0.0097,0.9921 ± 0.0097,0.9921 ± 0.0097,0.9960 ± 0.0080
test_f1,0.8323 ± 0.0312,0.8214 ± 0.0445,0.8373 ± 0.0263,0.7808 ± 0.0926
train_f1,0.8879 ± 0.0145,0.9043 ± 0.0126,0.9043 ± 0.0126,0.9029 ± 0.0090


In [32]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.7228 ± 0.0718,0.5136 ± 0.1044,0.6351 ± 0.0582,0.6972 ± 0.0935
train_roc_auc,0.9057 ± 0.0206,0.9960 ± 0.0032,0.9982 ± 0.0018,0.9973 ± 0.0029
test_precision,0.7822 ± 0.0743,0.7630 ± 0.0272,0.8151 ± 0.0422,0.7918 ± 0.0708
train_precision,0.8549 ± 0.0267,0.9407 ± 0.0308,0.9621 ± 0.0234,0.9693 ± 0.0255
test_recall,0.8744 ± 0.1052,0.9205 ± 0.0718,0.7949 ± 0.1493,0.8397 ± 0.0550
train_recall,0.9325 ± 0.0206,0.9960 ± 0.0080,0.9960 ± 0.0080,0.9881 ± 0.0097
test_f1,0.8196 ± 0.0506,0.8334 ± 0.0399,0.7933 ± 0.0680,0.8144 ± 0.0600
train_f1,0.8918 ± 0.0199,0.9674 ± 0.0195,0.9786 ± 0.0129,0.9785 ± 0.0167


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5459 ± 0.0704,0.5359 ± 0.1080,0.5179 ± 0.0532,0.4900 ± 0.0974
train_roc_auc,0.7977 ± 0.0135,0.8253 ± 0.0513,0.8253 ± 0.0456,0.8371 ± 0.0577
test_precision,0.7658 ± 0.0338,0.7794 ± 0.0469,0.7388 ± 0.0488,0.7450 ± 0.0298
train_precision,0.8527 ± 0.0310,0.8826 ± 0.0530,0.8999 ± 0.0395,0.8750 ± 0.0408
test_recall,0.9372 ± 0.0580,0.8397 ± 0.0550,0.7462 ± 0.1651,0.8423 ± 0.1184
train_recall,0.9523 ± 0.0349,0.9369 ± 0.0520,0.9206 ± 0.0674,0.9762 ± 0.0232
test_f1,0.8423 ± 0.0383,0.8080 ± 0.0467,0.7365 ± 0.1095,0.7871 ± 0.0665
train_f1,0.8988 ± 0.0161,0.9061 ± 0.0149,0.9072 ± 0.0170,0.9217 ± 0.0140


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.7005 ± 0.1222,0.5833 ± 0.1321,0.6518 ± 0.1454,0.6190 ± 0.1013
train_roc_auc,0.9585 ± 0.0095,0.9735 ± 0.0134,0.9785 ± 0.0043,0.9969 ± 0.0026
test_precision,0.7500 ± 0.0228,0.7375 ± 0.0338,0.7382 ± 0.0270,0.7454 ± 0.0331
train_precision,0.7876 ± 0.0101,0.8240 ± 0.0246,0.8264 ± 0.0151,0.8158 ± 0.0183
test_recall,1.0000 ± 0.0000,0.9833 ± 0.0333,0.9846 ± 0.0308,0.9692 ± 0.0615
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.8569 ± 0.0151,0.8427 ± 0.0322,0.8432 ± 0.0191,0.8407 ± 0.0217
train_f1,0.8811 ± 0.0063,0.9033 ± 0.0147,0.9049 ± 0.0090,0.8985 ± 0.0112
